# Dataflow examples

The six worked examples in `docs/dataflow_example_*.puml`, run here. Each diagram follows one of
these calls step by step, with a grid of the filled sites after each generation stage; this
notebook runs the same calls and shows the datasets they write.

Compile a diagram to PDF with `plantuml -tpdf docs/dataflow_example_single_var.puml`, or all of
them with `plantuml -tpdf docs/dataflow*.puml`.

Every case writes to its own folder under `./dataflow_examples`. Grids with three dimensions are
shown one `x0` slice per figure.

In [ ]:
import numpy as np

from tabray.notebook_helpers import run_case

BASE_DIR = "./dataflow_examples"

## Example 1: one variable on a 3x3 grid

`docs/dataflow_example_single_var.puml`. The same call as tutorial 1, "somehow_sparse_grid":
an LHS stage puts one site in every row and column, then each row (stratum) draws one fill site.

In [ ]:
ds, df = run_case(
    "single_var",
    num_obs=6,
    density=6 / 9,
    seed=12345,
    base_dir=BASE_DIR,
    title="Example 1: one variable",
)

## Example 2: two variables, one of them 1D

`docs/dataflow_example_multi_var.puml`. The same call as tutorial 3, "overlap_le1_1d". var1
varies along `x1` only and is pinned to one `x0`. Its density is a share of its own 3-cell axis,
so it holds 3 values, and the overlap target of 1/8 cannot be met: var0 covers every `x1`.

In [ ]:
ds, df = run_case(
    "multi_var",
    num_obs=8,
    density=[8 / 9, 8 / 9],
    num_vars=2,
    var_dims=[2, 1],
    overlap=1 / 8,
    fixed_overlap=False,
    seed=12345,
    base_dir=BASE_DIR,
    title="Example 2: var1 on x1 only",
)

## Example 3: fixed_overlap

`docs/dataflow_example_fixed_overlap.puml`. var1 and var2 both opt in to `fixed_overlap`, so they
take their overlap cells from one shared ordering of var0's sites: var1's cells on var0 are a
subset of var2's.

In [ ]:
ds, df = run_case(
    "fixed_overlap",
    num_obs=8,
    density=0.5,
    num_vars=3,
    var_dims=2,
    overlap=[0.5, 0.75],
    fixed_overlap=[True, True],
    seed=7,
    base_dir=BASE_DIR,
    title="Example 3: fixed_overlap=[True, True]",
)

The same call with `fixed_overlap=[False, False]`: the overlaps against var0 are the same, but
each variable picks its cells on var0 with its own stream.

In [ ]:
ds_free, df_free = run_case(
    "fixed_overlap_off",
    num_obs=8,
    density=0.5,
    num_vars=3,
    var_dims=2,
    overlap=[0.5, 0.75],
    fixed_overlap=[False, False],
    seed=7,
    base_dir=BASE_DIR,
    title="Example 3: fixed_overlap=[False, False]",
)


def on_var0_shared(dataset):
    """Sites of var1 on var0 that var2 also holds, out of var1's sites on var0."""
    held = {name: dataset[name].notnull().values for name in dataset.data_vars}
    var1_on_var0 = held["var0"] & held["var1"]
    return int((var1_on_var0 & held["var2"]).sum()), int(var1_on_var0.sum())


print("fixed_overlap=[True, True]:   %d of %d" % on_var0_shared(ds))
print("fixed_overlap=[False, False]: %d of %d" % on_var0_shared(ds_free))

## Example 4: a variable that drops one of three dimensions

`docs/dataflow_example_drop_dim_3d.puml`. On a 3x3x3 grid, var1 varies along `x0` and `x1` and is
pinned to one `x2`. Overlap is measured on `(x0, x1)`: a var1 value counts as overlap when var0
holds any site along `x2` above it, so two var1 values count without sharing a site with var0.

In [ ]:
ds, df = run_case(
    "drop_dim_3d",
    num_obs=12,
    density=12 / 27,
    num_vars=2,
    var_dims=[3, 2],
    overlap=0.5,
    seed=1,
    num_dims=3,
    base_dir=BASE_DIR,
    title="Example 4",
)

## Example 5: layout="padded" (experimental)

`docs/dataflow_example_padded.puml`. Each line (one `(x0, x1)` pair) fills its first k cells of the
padded axis `x2`. There is no LHS stage: coverage comes from every line holding `x2 = 0` and one
line running the full length. var1 follows the same layout, so its overlap with var0 is the ratio
of the two counts.

In [ ]:
ds, df = run_case(
    "padded",
    num_obs=18,
    density=[0.5, 0.25],
    num_vars=2,
    var_dims=3,
    seed=1234,
    num_dims=3,
    ratio_dims=[3, 3, 4],
    layout="padded",
    base_dir=BASE_DIR,
    title="Example 5",
)

## Example 6: the parallel path (experimental)

`docs/dataflow_example_parallel.puml`. `max_obs=5` splits 10 observations into 2 chunks along
`x0`. Each worker generates only its own strata and writes its own netCDF file; the parquet
chunks are consolidated into one dataset. The workers start with the spawn method, which a
notebook supports; a script needs an `if __name__ == "__main__":` guard.

In [ ]:
ds_par, df_par = run_case(
    "parallel",
    num_obs=10,
    density=10 / 16,
    seed=12345,
    max_obs=5,
    base_dir=BASE_DIR,
    title="Example 6: two chunks",
)

The serial run of the same call writes the same data: the record and the parquet columns match
value for value.

In [ ]:
ds_ser, df_ser = run_case(
    "parallel_serial",
    num_obs=10,
    density=10 / 16,
    seed=12345,
    base_dir=BASE_DIR,
    title="Example 6: serial",
)

print(
    "netCDF records equal:",
    np.array_equal(ds_par["record"].values, ds_ser["record"].values, equal_nan=True),
)
print(
    "parquet columns equal:",
    df_par.reset_index(drop=True).equals(df_ser.reset_index(drop=True)),
)